# Лабораторная работа № 1.3 — Простые итерации и метод Зейделя

Вариант 6. 

Приводим систему к x=αx+β. В простых итерациях все координаты берутся из старого вектора; в методе Зейделя уже обновлённые координаты используются сразу.

## Исходные данные

Числа ниже соответствуют файлу input.txt этой работы.

In [1]:
import math
import numpy as np
import matplotlib.pyplot as plt
from typing import List

rows = [[23.0, -6.0, -5.0, 9.0], [8.0, 22.0, -2.0, 5.0], [7.0, -6.0, 18.0, -1.0], [3.0, 5.0, 5.0, -19.0], [232.0, -82.0, 202.0, -57.0], [1e-06]]
A, b, eps = rows[:-2], rows[-2], rows[-1][0]
print('Точность:', eps)

Точность: 1e-06


## Итерационные формулы

Норма α меньше единицы обеспечивает сходимость. Остановка использует оценку ошибки по разности двух последовательных приближений.

In [2]:
def infinity_norm(x):
    return max(abs(value) for value in x)


def iteration_coefficients(A, b, eps, max_iter):
    if eps <= 0 or max_iter < 1:
        raise ValueError("eps and max_iter must be positive")
    n = len(A)
    if any(A[i][i] == 0 for i in range(n)):
        raise ValueError("A zero diagonal entry prevents fixed-point iteration")
    alpha = [[-A[i][j] / A[i][i] if i != j else 0.0 for j in range(n)] for i in range(n)]
    beta = [b[i] / A[i][i] for i in range(n)]
    return alpha, beta


def matrix_vector_mult(A, x):
    n = len(A)
    m = len(x)
    return [sum(A[i][k] * x[k] for k in range(m)) for i in range(n)]


def converged(A, b, old, new, eps, q, numerator):
    delta = infinity_norm([a - b for a, b in zip(new, old)])
    if not all(math.isfinite(value) for value in new):
        raise RuntimeError("The iteration diverged to non-finite values")
    if q < 1:
        # A posteriori bound in the same vector/matrix infinity norm.
        return numerator / (1 - q) * delta <= eps
    # Without a contraction bound, require both a small step and residual.
    residual = infinity_norm([value - rhs for value, rhs in zip(matrix_vector_mult(A, new), b)])
    return delta <= eps and residual <= eps


def simple_iteration_method(A, b, eps, max_iter):
    alpha, beta = iteration_coefficients(A, b, eps, max_iter)
    q = max(sum(abs(value) for value in row) for row in alpha)
    x = beta[:]
    for iterations in range(1, max_iter + 1):
        new = [sum(alpha[i][j] * x[j] for j in range(len(A))) + beta[i] for i in range(len(A))]
        if converged(A, b, x, new, eps, q, q):
            return new, iterations
        x = new
    raise RuntimeError(f"Simple iteration did not converge in {max_iter} iterations")


def seidel_method(A, b, eps, max_iter):
    alpha, beta = iteration_coefficients(A, b, eps, max_iter)
    n = len(A)
    q = max(sum(abs(value) for value in row) for row in alpha)
    c_norm = max(sum(abs(alpha[i][j]) for j in range(i + 1, n)) for i in range(n))
    x = beta[:]
    for iterations in range(1, max_iter + 1):
        new = x[:]
        for i in range(n):
            # Updated components are used immediately, without a matrix inverse.
            new[i] = beta[i] + sum(alpha[i][j] * new[j] for j in range(i)) + sum(alpha[i][j] * x[j] for j in range(i + 1, n))
        if converged(A, b, x, new, eps, q, c_norm):
            return new, iterations
        x = new
    raise RuntimeError(f"Seidel iteration did not converge in {max_iter} iterations")

In [3]:
alpha, beta = iteration_coefficients(A, b, eps, 100)
q = max(sum(abs(v) for v in row) for row in alpha)
x_j, n_j = simple_iteration_method(A, b, eps, 100)
x_s, n_s = seidel_method(A, b, eps, 100)
print('||α||∞ =', q)
print('Простые итерации:', np.round(x_j, 8), 'шагов:', n_j)
print('Зейдель:', np.round(x_s, 8), 'шагов:', n_s)

||α||∞ = 0.8695652173913043
Простые итерации: [ 8.00000001 -6.99999994  6.00000004  3.99999996] шагов: 31
Зейдель: [ 8.00000001 -7.          5.99999999  4.        ] шагов: 10


## Самопроверка

Как проверить, что оба метода решают одну и ту же систему?

In [4]:
assert q < 1
for name, x in [('Итерации', x_j), ('Зейдель', x_s)]:
    residual = np.max(np.abs(np.array(A)@x-b))
    assert residual < 1e-4
    print(name, 'невязка:', residual)

Итерации невязка: 1.3320613945211335e-06
Зейдель невязка: 3.220051780772337e-07
